In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split

In [10]:
DATA_PATH = "diabetes.csv"
data = pd.read_csv(DATA_PATH)

In [11]:
# Cek missing values dan nilai 0 d
print("\nNilai 0 pada setiap kolom:")
jumlah_nol = {}
for kolom in data.columns:
    if kolom != 'Outcome':
        hitung_nol = (data[kolom] == 0).sum()
        jumlah_nol[kolom] = hitung_nol
        print(f"{kolom}: {hitung_nol}")

# Total baris yang memiliki minimal satu nilai 0 (selain Outcome dan Pregnancies)
fitur_dengan_nol = data.drop(['Outcome', 'Pregnancies'], axis=1) == 0
total_baris_nol = fitur_dengan_nol.any(axis=1).sum()
persentase_baris_nol = (total_baris_nol / len(data)) * 100
print(f"Total baris yang memiliki minimal satu nilai 0: {total_baris_nol}")
print(f"Persentase baris dengan nilai 0: {persentase_baris_nol:.2f}%")


Nilai 0 pada setiap kolom:
Pregnancies: 301
Glucose: 13
BloodPressure: 90
SkinThickness: 573
Insulin: 956
BMI: 28
DiabetesPedigreeFunction: 0
Age: 0
Total baris yang memiliki minimal satu nilai 0: 965
Persentase baris dengan nilai 0: 48.25%


In [12]:
kolom_hilang = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
data[kolom_hilang] = data[kolom_hilang].replace(0, np.nan)
if persentase_baris_nol > 10:
    X = data.drop(columns=["Outcome"])
    y = data["Outcome"].to_numpy()
    X_imputed = KNNImputer(n_neighbors=5).fit_transform(X)
    data = pd.DataFrame(X_imputed, columns=X.columns).assign(Outcome=y)
    print(f"Jumlah missing values setelah imputasi:\n{data.isnull().sum()}")
else:
    data = data.dropna(subset=kolom_hilang).reset_index(drop=True)
    print(f"Jumlah data setelah menghapus baris dengan nilai 0: {len(data)}")

Jumlah missing values setelah imputasi:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [13]:
X = data.drop(columns=["Outcome"])
y = data["Outcome"]

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [15]:
df_train_export = pd.concat([X_train, y_train], axis=1)
df_test_export = pd.concat([X_test, y_test], axis=1)

df_train_export.to_excel('data_training_mentah.xlsx', index=False)
df_test_export.to_excel('data_testing_mentah.xlsx', index=False)